# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_csv("../data/processed/rideshare_cleaned.csv")
dfe = df.copy()


In [5]:
tier_map = {
    'Shared': 'Economy', 'UberPool': 'Economy',
    'Lyft': 'Mid',       'UberX': 'Mid',    'WAV': 'Mid',
    'Lyft XL': 'Mid',    'UberXL': 'Mid',
    'Lux': 'Premium',    'Black': 'Premium',
    'Lux Black': 'Premium', 'Black SUV': 'Premium', 'Lux Black XL': 'Premium'
}
dfe['tier'] = dfe['name'].map(tier_map)


In [6]:
dfe['route'] = dfe['source'] + ' → ' + dfe['destination']

In [8]:
dfe['log_price']        = np.log1p(dfe['price'])                          # 0.9587
dfe['tier_enc']         = dfe['tier'].map({'Economy': 0, 'Mid': 1, 'Premium': 2})  # 0.7347

le = LabelEncoder()
dfe['name_enc']         = le.fit_transform(dfe['name'].astype(str))       # 0.5866

dfe['surge_x_distance'] = dfe['surge_multiplier'] * dfe['distance']      # 0.3868
# distance                                                                 # 0.3451
dfe['log_distance']     = np.log1p(dfe['distance'])                       # 0.3353
dfe['surge_intensity']  = (dfe['surge_multiplier'] - 1) * dfe['distance']# 0.2613
dfe['price_per_mile']   = dfe['price'] / (dfe['distance'] + 0.001)       # 0.2438
# surge_multiplier                                                         # 0.2405
dfe['is_surge']         = (dfe['surge_multiplier'] > 1).astype(int)      # 0.2233

dfe['source_avg_price'] = dfe['source'].map(dfe.groupby('source')['price'].mean())      # 0.1579
dfe['dest_avg_price']   = dfe['destination'].map(dfe.groupby('destination')['price'].mean())  # 0.1501

dfe['cab_type_enc']     = (dfe['cab_type'] == 'Lyft').astype(int)        # 0.0834

dfe['destination_enc']  = le.fit_transform(dfe['destination'].astype(str))  # 0.0463
dfe['route_frequency']  = dfe['route'].map(dfe['route'].value_counts())     # 0.0463
dfe['source_enc']       = le.fit_transform(dfe['source'].astype(str))       # 0.0254

# Export

In [10]:
import os

output = "../data/processed/rideshare_feature_engineering.csv"
os.makedirs(os.path.dirname(output), exist_ok=True)

dfe.to_csv(output, index=False)

print(f"บันทึกไฟล์เรียบร้อย: {output}")
print(f"ขนาดข้อมูลที่บันทึก: {dfe.shape[0]} แถว x {dfe.shape[1]} คอลัมน์")

บันทึกไฟล์เรียบร้อย: ../data/processed/rideshare_feature_engineering.csv
ขนาดข้อมูลที่บันทึก: 637976 แถว x 53 คอลัมน์
